In [ ]:
from pathlib import Path as _ProjectPath
import sys as _project_sys

_project_root = next(
    (
        p
        for p in (_ProjectPath.cwd(), *_ProjectPath.cwd().parents)
        if (p / "project_config.json").is_file()
    ),
    None,
)
if _project_root is None:
    raise FileNotFoundError(
        "Open VS Code in the organized project root before running this notebook."
    )
if str(_project_root / "tools") not in _project_sys.path:
    _project_sys.path.insert(0, str(_project_root / "tools"))
from project_paths import ProjectPaths as _ProjectPaths

_paths = _ProjectPaths(_project_root, 'rmhd_shap_lime_baseline')

import os, glob, json, gzip, random, re
from pathlib import Path
import pandas as pd
import numpy as np


DATA_DIR = _ProjectPath(_paths.rmhd_dir)
SAVE_DIR = _ProjectPath(_paths.output_dir)
SAVE_DIR.mkdir(parents=True, exist_ok=True)


MENTAL_HEALTH_SUBS = {
    'EDAnonymous',
    'addiction',
    'alcoholism',
    'adhd',
    'anxiety',
    'autism',
    'bipolarreddit',
    'bpd',
    'depression',
    'healthanxiety',
    'lonely',
    'ptsd',
    'schizophrenia',
    'socialanxiety',
    'suicidewatch',
    'mentalhealth',
    'COVID19_support',
}
CONTROL_SUBS = {
    'conspiracy',
    'divorce',
    'fitness',
    'guns',
    'jokes',
    'legaladvice',
    'meditation',
    'parenting',
    'personalfinance',
    'relationships',
    'teaching',
}


MIN_PER_CLASS = 500
RANDOM_STATE = 42


In [ ]:
def _read_csv_flex(path: Path) -> pd.DataFrame:
    df = pd.read_csv(_paths.input_path(path))
    return df


def _read_jsonl_flex(path: Path) -> pd.DataFrame:
    records = []
    open_fn = gzip.open if path.suffix == '.gz' else open
    with open_fn(path, 'rt', encoding='utf-8') as f:
        for line in f:
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    return pd.DataFrame.from_records(records)


def _coalesce_text_columns(df: pd.DataFrame) -> pd.DataFrame:

    cand_title = [c for c in df.columns if c.lower() in {'title'}]
    cand_body = [c for c in df.columns if c.lower() in {'selftext', 'body', 'text'}]

    title = df[cand_title[0]].astype(str) if cand_title else ""
    body = df[cand_body[0]].astype(str) if cand_body else ""
    if isinstance(title, str):
        text = body
    elif isinstance(body, str):
        text = title
    else:
        text = (title.fillna("") + "\n" + body.fillna("")).str.strip()

    sub_cols = [c for c in df.columns if c.lower() == 'subreddit']
    if sub_cols:
        subreddit = (
            df[sub_cols[0]].astype(str).str.replace(r'^r/', '', regex=True).str.lower()
        )
    else:

        alt = [c for c in df.columns if c.lower() in {'community', 'sub'}]
        subreddit = (
            df[alt[0]].astype(str).str.lower()
            if alt
            else pd.Series(['unknown'] * len(df))
        )

    return pd.DataFrame({'text': text, 'subreddit': subreddit})


def load_rmhd_text(data_dir: Path) -> pd.DataFrame:
    files = []
    for ext in ('*.csv', '*.csv.gz', '*.jsonl', '*.jsonl.gz', '*.json', '*.json.gz'):
        files += glob.glob(str(data_dir / '**' / ext), recursive=True)
    if not files:
        raise FileNotFoundError(
            f"No files found under {data_dir}. Put RMHD text files there."
        )
    frames = []
    for fp in files:
        p = Path(fp)
        try:
            if (
                p.suffix in {'.csv', '.gz'}
                and p.name.endswith('.csv')
                or p.name.endswith('.csv.gz')
            ):
                df = _read_csv_flex(p)
            elif (
                p.suffix in {'.jsonl', '.json'}
                or p.name.endswith('.jsonl.gz')
                or p.name.endswith('.json.gz')
            ):
                df = _read_jsonl_flex(p)
            else:
                continue
            frames.append(_coalesce_text_columns(df))
        except Exception as e:
            print(f"[WARN] Skipping {p.name}: {e}")
            continue
    all_df = pd.concat(frames, ignore_index=True).dropna(subset=['text', 'subreddit'])

    all_df['subreddit'] = (
        all_df['subreddit'].str.replace(r'^r/', '', regex=True).str.lower()
    )
    return all_df


df = load_rmhd_text(DATA_DIR)
print("Loaded:", df.shape, "columns:", list(df.columns)[:5])
df.head(2)


In [ ]:
known = {*(s.lower() for s in MENTAL_HEALTH_SUBS), *(s.lower() for s in CONTROL_SUBS)}
df = df[df['subreddit'].isin(known)].copy()


df['text'] = df['text'].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()


counts = df['subreddit'].value_counts()
keep_classes = counts[counts >= MIN_PER_CLASS].index.tolist()
if len(keep_classes) < len(counts):
    print("Dropping rare classes:", sorted(set(counts.index) - set(keep_classes)))
df = df[df['subreddit'].isin(keep_classes)].copy()

print("Final class counts:")
print(df['subreddit'].value_counts())


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

X_text = df['text'].tolist()
y_labels = df['subreddit'].tolist()

le = LabelEncoder()
y = le.fit_transform(y_labels)

X_train, X_tmp, y_train, y_tmp = train_test_split(
    X_text, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.5, random_state=RANDOM_STATE, stratify=y_tmp
)

print(f"Train/Val/Test sizes: {len(X_train)}/{len(X_val)}/{len(X_test)}")
print("Classes:", list(le.classes_))


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, f1_score

pipeline = Pipeline(
    [
        (
            "tfidf",
            TfidfVectorizer(
                ngram_range=(1, 2),
                min_df=5,
                max_df=0.9,
                strip_accents='unicode',
                lowercase=True,
                max_features=200000,
            ),
        ),
        (
            "clf",
            LogisticRegression(
                solver='saga',
                penalty='l2',
                C=2.0,
                max_iter=300,
                n_jobs=-1,
                multi_class='ovr',
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

pipeline.fit(X_train, y_train)
pred_val = pipeline.predict(X_val)

print("Validation macro F1:", f1_score(y_val, pred_val, average='macro'))
print("Validation micro F1:", f1_score(y_val, pred_val, average='micro'))
print("\nDetailed report (val):")
print(classification_report(y_val, pred_val, target_names=le.classes_))


In [ ]:
pred_test = pipeline.predict(X_test)
print("Test macro F1:", f1_score(y_test, pred_test, average='macro'))
print("Test micro F1:", f1_score(y_test, pred_test, average='micro'))
print("\nDetailed report (test):")
print(classification_report(y_test, pred_test, target_names=le.classes_))


In [ ]:
import joblib

joblib.dump(pipeline, _paths.output_path(SAVE_DIR / "rmhd_tfidf_logreg.joblib"))
joblib.dump(le, _paths.output_path(SAVE_DIR / "label_encoder.joblib"))
print("Saved to", SAVE_DIR.resolve())


In [ ]:
from lime.lime_text import LimeTextExplainer

class_names = list(le.classes_)
explainer = LimeTextExplainer(class_names=class_names)

idx = random.randint(0, len(X_test) - 1)
sample_text = X_test[idx]
sample_true = class_names[y_test[idx]]

print("Sample true class:", sample_true)
print("\nTEXT:\n", sample_text[:1000], "..." if len(sample_text) > 1000 else "")


def predict_proba(texts):

    return pipeline.predict_proba(texts)


exp = explainer.explain_instance(
    sample_text, predict_proba, num_features=10, labels=[0]
)
try:

    exp.show_in_notebook(text=True)
except Exception as e:
    print("LIME weights (top 10):", exp.as_list())


In [ ]:
import shap


bg_size = min(200, len(X_train))
bg_texts = X_train[:bg_size]


vec = pipeline.named_steps['tfidf']
clf = pipeline.named_steps['clf']


X_bg = vec.transform(bg_texts)
x_sample = vec.transform([sample_text])

explainer = shap.LinearExplainer(clf, X_bg, feature_dependence="independent")
shap_values = explainer.shap_values(x_sample)


pred_idx = int(pipeline.predict([sample_text])[0])
pred_class = class_names[pred_idx]


feature_names = np.array(vec.get_feature_names_out())

row = shap_values[pred_idx].toarray()[0]
topk = 15
top_pos_idx = np.argsort(-row)[:topk]
top_neg_idx = np.argsort(row)[:topk]

print(f"Top + contributors for class '{pred_class}':")
for i in top_pos_idx:
    if row[i] > 0:
        print(f"{feature_names[i]}\t+{row[i]:.4f}")
print(f"\nTop - contributors (pushing away from '{pred_class}')")
for i in top_neg_idx:
    if row[i] < 0:
        print(f"{feature_names[i]}\t{row[i]:.4f}")


In [ ]:
def predict_with_explanations(text: str, top_k=10):
    pred_proba = pipeline.predict_proba([text])[0]
    pred_idx = int(np.argmax(pred_proba))
    pred_label = class_names[pred_idx]

    try:
        exp = explainer.explain_instance(
            text, predict_proba, num_features=top_k, labels=[pred_idx]
        )
        lime_weights = exp.as_list(label=pred_idx)
    except Exception:
        lime_weights = []

    x = vec.transform([text])
    shap_vals = explainer.shap_values(x)
    row = shap_vals[pred_idx].toarray()[0]
    feature_names = np.array(vec.get_feature_names_out())
    top_idx = np.argsort(-np.abs(row))[:top_k]
    shap_pairs = [(feature_names[i], float(row[i])) for i in top_idx]
    return pred_label, float(pred_proba[pred_idx]), lime_weights, shap_pairs


pred_label, pred_conf, lime_w, shap_w = predict_with_explanations(sample_text, top_k=10)
print("Pred:", pred_label, f"({pred_conf:.3f})")
print("LIME top terms:", lime_w[:5])
print("SHAP top terms:", shap_w[:5])



## Notes, Reproducibility & Ethics

- **Files organization:** Put all RMHD *posts* CSV/JSONL files under `data/rmhd/`. The loader recursively scans subfolders and merges them.
- **Explainability fidelity:** With TF‑IDF + Logistic Regression, both **LIME** and **SHAP** map back to actual **words/ngrams**, making it easy to interpret *hashtags/keywords*.
- **Speed tips:** Reduce `max_features` in the vectorizer if memory is limited; or set `MIN_PER_CLASS` lower if your subset is small.
- **Re-use:** Artifacts are saved under `artifacts/` (pipeline + label encoder).
- **Ethics:** Do **not** use these models for clinical decision-making. If your work surfaces self-harm content, follow your institution's safety protocols.
